# 00 — UK-DALE Data Exploration & Visualization

Complete visual analysis of the UK-DALE dataset:
- Aggregate power signal analysis
- Per-appliance power distributions
- Daily and weekly usage patterns
- DWT frequency decomposition visualization
- Spectrogram / time-frequency representation
- Cross-house comparison (House 1 vs House 2)
- Class imbalance analysis
- Power signature visualization per appliance

**Author:** Chadha Jeddi — NILM Benchmarking Project

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, '../src')
sys.path.insert(0, '../models')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import seaborn as sns
import pywt
import warnings
warnings.filterwarnings('ignore')

from config import APPLIANCE_NAMES, APPLIANCES, WINDOW_SIZE
from preprocessing import load_ukdale_house, preprocess_house
from dataset import load_clean_df, save_clean_df

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

COLORS = {
    'kettle':          '#D94040',
    'fridge':          '#2E9E5A',
    'washing_machine': '#E8922A',
    'dishwasher':      '#7B4FBF',
    'microwave':       '#CC3399',
    'aggregate':       '#3366CC',
}

print('Setup complete')
print(f'Appliances: {APPLIANCE_NAMES}')


## 2. Load UK-DALE Data

In [ ]:
# Load House 1 and House 2
print('Loading UK-DALE House 1...')
cached1 = load_clean_df('UK-DALE', 1)
if cached1 is not None:
    h1 = cached1
else:
    raw1 = load_ukdale_house(house=1)
    h1 = preprocess_house(raw1)
    save_clean_df(h1, 'UK-DALE', 1)

print('Loading UK-DALE House 2...')
cached2 = load_clean_df('UK-DALE', 2)
if cached2 is not None:
    h2 = cached2
else:
    raw2 = load_ukdale_house(house=2)
    h2 = preprocess_house(raw2)
    save_clean_df(h2, 'UK-DALE', 2)

print(f'House 1: {len(h1):,} rows | {(h1.index[-1]-h1.index[0]).days} days')
print(f'House 2: {len(h2):,} rows | {(h2.index[-1]-h2.index[0]).days} days')
print(f'Columns: {list(h1.columns)}')
h1.head(3)


## 3. Basic Statistics

In [ ]:
print('='*70)
print('HOUSE 1 — APPLIANCE STATISTICS')
print('='*70)
print(f'{"Appliance":<22} {"Mean(W)":>8} {"Max(W)":>8} {"Duty%":>7} {"ON events":>10}')
print('-'*60)
for a in APPLIANCE_NAMES:
    mean_w = h1[a].mean()
    max_w  = h1[a].max()
    duty   = h1[f'{a}_state'].mean() * 100
    events = int((h1[f'{a}_state'].diff() == 1).sum())
    print(f'{a:<22} {mean_w:>8.1f} {max_w:>8.0f} {duty:>7.2f} {events:>10,}')

print()
print(f'Aggregate: mean={h1["aggregate"].mean():.1f}W  '
      f'max={h1["aggregate"].max():.0f}W  '
      f'std={h1["aggregate"].std():.1f}W')


## 4. Full Day Overview
Aggregate + all appliances for one representative day.

In [ ]:
# Select a representative day with all appliances active
day_data = None
for date in pd.date_range(h1.index[0].date(), h1.index[-1].date()):
    day = h1[h1.index.date == date.date()]
    if len(day) < 100: continue
    active = sum(day[f'{a}_state'].max() > 0 for a in APPLIANCE_NAMES)
    if active >= 4:
        day_data = day
        break

fig, axes = plt.subplots(len(APPLIANCE_NAMES)+1, 1,
                          figsize=(18, 3*(len(APPLIANCE_NAMES)+1)),
                          sharex=True)

# Aggregate
axes[0].plot(day_data.index, day_data['aggregate'],
             color=COLORS['aggregate'], linewidth=0.8)
axes[0].fill_between(day_data.index, day_data['aggregate'],
                      alpha=0.3, color=COLORS['aggregate'])
axes[0].set_ylabel('Aggregate (W)', fontsize=10)
axes[0].set_title(f'UK-DALE House 1 — Full Day Overview ({day_data.index[0].date()})',
                  fontsize=13)

# Per appliance
for i, a in enumerate(APPLIANCE_NAMES):
    ax = axes[i+1]
    ax.plot(day_data.index, day_data[a],
            color=COLORS[a], linewidth=1.2, label='Power (W)')
    ax.fill_between(day_data.index, day_data[a],
                    alpha=0.35, color=COLORS[a])
    # State overlay
    on_mask = day_data[f'{a}_state'] > 0
    ax.fill_between(day_data.index,
                    ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 0,
                    day_data[a].max()*1.1,
                    where=on_mask, alpha=0.1, color=COLORS[a])
    max_w = APPLIANCES[a]['max_power']
    duty = day_data[f'{a}_state'].mean()*100
    ax.set_ylabel(f'{a.replace("_"," ").title()}\n(W)', fontsize=9)
    ax.text(0.01, 0.85, f'duty={duty:.1f}%',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[-1].set_xlabel('Time', fontsize=10)
plt.tight_layout()
plt.savefig('../experiments/results/00_full_day_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 5. Power Distribution Per Appliance

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Aggregate
agg_vals = h1['aggregate'].values
axes[0].hist(agg_vals[agg_vals > 0], bins=100,
             color=COLORS['aggregate'], alpha=0.7, edgecolor='white')
axes[0].set_title('Aggregate Power Distribution', fontsize=11)
axes[0].set_xlabel('Power (W)')
axes[0].set_ylabel('Count')
axes[0].axvline(agg_vals.mean(), color='red', linestyle='--',
                label=f'Mean={agg_vals.mean():.0f}W')
axes[0].legend()

# Per appliance — ON state only
for i, a in enumerate(APPLIANCE_NAMES):
    ax = axes[i+1]
    on_vals = h1.loc[h1[f'{a}_state'] > 0, a].values
    if len(on_vals) > 0:
        ax.hist(on_vals, bins=80, color=COLORS[a], alpha=0.75,
                edgecolor='white')
        ax.axvline(on_vals.mean(), color='black', linestyle='--',
                   label=f'Mean={on_vals.mean():.0f}W')
        ax.set_title(f'{a.replace("_"," ").title()}\n(when ON only)',
                     fontsize=10)
        ax.set_xlabel('Power (W)')
        ax.legend(fontsize=9)
    else:
        ax.text(0.5, 0.5, 'No ON events', ha='center', va='center',
                transform=ax.transAxes)

plt.suptitle('UK-DALE House 1 — Power Distributions (ON state)', fontsize=13)
plt.tight_layout()
plt.savefig('../experiments/results/00_power_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 6. Hourly Usage Heatmap (When Are Appliances Used?)

In [ ]:
fig, axes = plt.subplots(1, N_APP := len(APPLIANCE_NAMES),
                          figsize=(4*N_APP, 6))

for i, a in enumerate(APPLIANCE_NAMES):
    # Build hour x day-of-week matrix
    df_tmp = h1.copy()
    df_tmp['hour'] = df_tmp.index.hour
    df_tmp['dow']  = df_tmp.index.dayofweek
    pivot = df_tmp.groupby(['hour','dow'])[f'{a}_state'].mean().unstack()
    pivot.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

    sns.heatmap(pivot, ax=axes[i], cmap='YlOrRd',
                cbar=i==N_APP-1, vmin=0, vmax=pivot.values.max())
    axes[i].set_title(a.replace('_',' ').title(), fontsize=10)
    axes[i].set_xlabel('Day of week')
    if i == 0:
        axes[i].set_ylabel('Hour of day')
    else:
        axes[i].set_ylabel('')

plt.suptitle('UK-DALE House 1 — Appliance Usage Heatmap\n'
             '(fraction of time ON per hour × day-of-week)',
             fontsize=13)
plt.tight_layout()
plt.savefig('../experiments/results/00_hourly_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 7. Class Imbalance Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Duty cycle bar chart
duties = [h1[f'{a}_state'].mean()*100 for a in APPLIANCE_NAMES]
colors = [COLORS[a] for a in APPLIANCE_NAMES]
bars = axes[0].bar(APPLIANCE_NAMES, duties, color=colors, alpha=0.85)
axes[0].set_title('Duty Cycle — Fraction of Time ON (%)', fontsize=11)
axes[0].set_ylabel('%')
axes[0].set_xticklabels(APPLIANCE_NAMES, rotation=20, ha='right')
for bar, duty in zip(bars, duties):
    axes[0].text(bar.get_x()+bar.get_width()/2, duty+0.1,
                 f'{duty:.2f}%', ha='center', fontsize=10)

# ON/OFF imbalance pie charts
n_app = len(APPLIANCE_NAMES)
cols = n_app
inner_ax = axes[1]
inner_ax.axis('off')
inner_fig, inner_axes = plt.subplots(1, n_app, figsize=(3*n_app, 3))
for i, a in enumerate(APPLIANCE_NAMES):
    on  = h1[f'{a}_state'].sum()
    off = len(h1) - on
    inner_axes[i].pie([off, on],
                      labels=['OFF', 'ON'],
                      colors=['#EEEEEE', COLORS[a]],
                      autopct='%1.1f%%',
                      startangle=90,
                      textprops={'fontsize': 9})
    inner_axes[i].set_title(a.replace('_',' ').title(), fontsize=9)
inner_fig.suptitle('ON/OFF Class Imbalance per Appliance', fontsize=12)
inner_fig.tight_layout()
inner_fig.savefig('../experiments/results/00_class_imbalance_pie.png',
                  dpi=150, bbox_inches='tight')

plt.figure(fig.number)
plt.tight_layout()
plt.savefig('../experiments/results/00_class_imbalance.png',
            dpi=150, bbox_inches='tight')
plt.show()
inner_fig.show()


## 8. Power Signatures — Typical Appliance Cycles
Shows what each appliance looks like during a typical activation event.

In [ ]:
fig, axes = plt.subplots(1, len(APPLIANCE_NAMES),
                          figsize=(5*len(APPLIANCE_NAMES), 5))

for i, a in enumerate(APPLIANCE_NAMES):
    ax = axes[i]
    max_w = APPLIANCES[a]['max_power']

    # Find ON events
    state = h1[f'{a}_state'].values
    power = h1[a].values
    diffs = np.diff(state)
    on_starts = np.where(diffs == 1)[0]

    # Plot up to 5 typical events
    WIN = 300  # 30 minutes window
    n_plotted = 0
    for start in on_starts:
        if start + WIN > len(power): continue
        event = power[start:start+WIN]
        if event.max() < APPLIANCES[a]['power_threshold']: continue
        t = np.arange(WIN) * 6 / 60  # minutes
        ax.plot(t, event, color=COLORS[a], alpha=0.4, linewidth=1.0)
        n_plotted += 1
        if n_plotted >= 10: break

    # Mean signature
    signatures = []
    for start in on_starts:
        if start + WIN > len(power): continue
        event = power[start:start+WIN]
        if event.max() > APPLIANCES[a]['power_threshold']:
            signatures.append(event)
    if signatures:
        mean_sig = np.mean(signatures, axis=0)
        t = np.arange(WIN) * 6 / 60
        ax.plot(t, mean_sig, color=COLORS[a], linewidth=2.5,
                label='Mean signature')

    ax.set_title(f'{a.replace("_"," ").title()}\n({len(signatures)} events)',
                 fontsize=10)
    ax.set_xlabel('Time (minutes)')
    if i == 0: ax.set_ylabel('Power (W)')
    ax.legend(fontsize=8)

plt.suptitle('UK-DALE House 1 — Typical Appliance Power Signatures',
             fontsize=13)
plt.tight_layout()
plt.savefig('../experiments/results/00_power_signatures.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 9. DWT Decomposition Visualization
Shows how the Discrete Wavelet Transform decomposes the aggregate signal.

In [ ]:
# Select a window with kettle activation
kettle_on = h1[h1['kettle_state'] > 0]
if len(kettle_on) > WINDOW_SIZE:
    start_idx = h1.index.get_loc(kettle_on.index[100])
    start_idx = max(0, start_idx - WINDOW_SIZE//2)
    window = h1['aggregate'].values[start_idx:start_idx+WINDOW_SIZE].astype(float)
else:
    window = h1['aggregate'].values[:WINDOW_SIZE].astype(float)

# DWT decomposition
coeffs = pywt.wavedec(window, 'db4', level=3)
cA3, cD3, cD2, cD1 = coeffs

# Reconstruct each level
def reconstruct_level(coeffs, level, wavelet='db4'):
    zero_coeffs = [np.zeros_like(c) for c in coeffs]
    zero_coeffs[level] = coeffs[level]
    return pywt.waverec(zero_coeffs, wavelet)[:WINDOW_SIZE]

LP3 = pywt.waverec([cA3, None, None, None], 'db4')[:WINDOW_SIZE]
HP3 = reconstruct_level(coeffs, 1, 'db4')
HP2 = reconstruct_level(coeffs, 2, 'db4')
HP1 = reconstruct_level(coeffs, 3, 'db4')

t = np.arange(WINDOW_SIZE) * 6  # seconds

fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True)

# Original signal
axes[0].plot(t, window, color='#3366CC', linewidth=1.0)
axes[0].fill_between(t, window, alpha=0.3, color='#3366CC')
axes[0].set_ylabel('Original\nAggregate (W)', fontsize=10)
axes[0].set_title('DWT Decomposition — Aggregate Power Signal (db4 wavelet, 3 levels)',
                  fontsize=12)

# DWT bands
band_info = [
    (LP3, '#2E9E5A', 'LP3\n(Trend/Baseline)'),
    (HP3, '#E8922A', 'HP3\n(Slow variations)'),
    (HP2, '#7B4FBF', 'HP2\n(Medium transients)'),
    (HP1, '#D94040', 'HP1\n(Fast transients)'),
]
for j, (band, color, label) in enumerate(band_info):
    axes[j+1].plot(t, band, color=color, linewidth=1.0)
    axes[j+1].fill_between(t, band, 0, alpha=0.3, color=color)
    axes[j+1].set_ylabel(label, fontsize=9)
    axes[j+1].axhline(0, color='gray', linewidth=0.5)

axes[-1].set_xlabel('Time (seconds)', fontsize=10)
plt.tight_layout()
plt.savefig('../experiments/results/00_dwt_decomposition.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 10. Spectrogram — Time-Frequency Representation
Shows the frequency content of the aggregate signal over time.

In [ ]:
from scipy import signal as scipy_signal

# Use the same window as DWT
fig, axes = plt.subplots(len(APPLIANCE_NAMES)+1, 1,
                          figsize=(16, 4*(len(APPLIANCE_NAMES)+1)))

def plot_spectrogram(ax, sig, title, fs=1/6):
    f, t_spec, Sxx = scipy_signal.spectrogram(
        sig, fs=fs, nperseg=32, noverlap=24)
    im = ax.pcolormesh(t_spec, f*1000, 10*np.log10(Sxx+1e-10),
                       shading='gouraud', cmap='inferno')
    ax.set_ylabel('Freq (mHz)', fontsize=9)
    ax.set_title(title, fontsize=10)
    return im

# Find a good window — all appliances active
best_start = 0
WIN_SPEC = 2880  # 8 hours
for s in range(0, len(h1)-WIN_SPEC, WIN_SPEC):
    active = sum(h1.iloc[s:s+WIN_SPEC][f'{a}_state'].max() > 0
                 for a in APPLIANCE_NAMES)
    if active >= 4:
        best_start = s; break

agg_window = h1['aggregate'].values[best_start:best_start+WIN_SPEC]
im = plot_spectrogram(axes[0], agg_window,
                       'Aggregate Power — Spectrogram')

for i, a in enumerate(APPLIANCE_NAMES):
    app_window = h1[a].values[best_start:best_start+WIN_SPEC]
    plot_spectrogram(axes[i+1], app_window,
                     f'{a.replace("_"," ").title()} — Spectrogram')

axes[-1].set_xlabel('Time (seconds)', fontsize=10)
plt.colorbar(im, ax=axes, label='Power (dB)', shrink=0.6)
plt.suptitle('UK-DALE House 1 — Spectrogram Representation\n'
             '(Time-Frequency Analysis)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../experiments/results/00_spectrogram.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 11. RGB Image Representation
Maps DWT sub-bands to RGB channels — visualizes the power signal as an image.
Red=LP3 (trend), Green=HP2 (cycles), Blue=HP1 (transients)

In [ ]:
from dwt import dwt_transform

# Select 20 random windows — 10 for each appliance ON/OFF
N_EXAMPLES = 5
fig, axes = plt.subplots(
    len(APPLIANCE_NAMES)+1, N_EXAMPLES*2,
    figsize=(N_EXAMPLES*4, 3*(len(APPLIANCE_NAMES)+1))
)

def window_to_rgb(window_1d):
    """Convert 1D power window to RGB image via DWT."""
    dwt = dwt_transform(window_1d.astype(np.float32))  # (4, 480)
    # Normalize each channel independently
    def norm(x):
        mn, mx = x.min(), x.max()
        return (x - mn) / (mx - mn + 1e-8)
    R = norm(dwt[0])  # LP3 — trend
    G = norm(dwt[2])  # HP2 — cycles
    B = norm(dwt[3])  # HP1 — transients
    # Reshape 1D (480) to 2D (24, 20)
    R = R.reshape(24, 20)
    G = G.reshape(24, 20)
    B = B.reshape(24, 20)
    return np.stack([R, G, B], axis=-1)  # (24, 20, 3)

# Aggregate examples
agg_vals = h1['aggregate'].values
for j in range(N_EXAMPLES*2):
    start = np.random.randint(0, len(agg_vals)-WINDOW_SIZE)
    window = agg_vals[start:start+WINDOW_SIZE]
    rgb = window_to_rgb(window)
    axes[0, j].imshow(rgb, aspect='auto')
    axes[0, j].axis('off')
    if j == 0:
        axes[0, j].set_title('Aggregate\nON', fontsize=8)

# Per appliance — ON and OFF examples
for i, a in enumerate(APPLIANCE_NAMES):
    on_idx  = h1.index[h1[f'{a}_state'] > 0]
    off_idx = h1.index[h1[f'{a}_state'] == 0]

    for j in range(N_EXAMPLES):
        # ON example
        if len(on_idx) > WINDOW_SIZE:
            start = h1.index.get_loc(on_idx[j*100 % len(on_idx)])
            start = max(0, min(start, len(agg_vals)-WINDOW_SIZE))
            window = agg_vals[start:start+WINDOW_SIZE]
            rgb = window_to_rgb(window)
            axes[i+1, j].imshow(rgb, aspect='auto')
            axes[i+1, j].axis('off')
            if j == 0:
                axes[i+1, j].set_ylabel(
                    a.replace('_',' ').title(), fontsize=8)
            if i == 0 and j == 0:
                axes[i+1, j].set_title('ON examples', fontsize=8)

        # OFF example
        if len(off_idx) > WINDOW_SIZE:
            start = h1.index.get_loc(off_idx[j*100 % len(off_idx)])
            start = max(0, min(start, len(agg_vals)-WINDOW_SIZE))
            window = agg_vals[start:start+WINDOW_SIZE]
            rgb = window_to_rgb(window)
            axes[i+1, N_EXAMPLES+j].imshow(rgb, aspect='auto')
            axes[i+1, N_EXAMPLES+j].axis('off')
            if i == 0 and j == 0:
                axes[i+1, N_EXAMPLES+j].set_title('OFF examples', fontsize=8)

# Dividing line between ON and OFF
fig.add_artist(plt.Line2D(
    [N_EXAMPLES/(N_EXAMPLES*2), N_EXAMPLES/(N_EXAMPLES*2)],
    [0, 1], transform=fig.transFigure,
    color='red', linewidth=2, linestyle='--'))

plt.suptitle(
    'RGB Image Representation of Power Windows\n'
    'R=LP3(trend)  G=HP2(cycles)  B=HP1(transients)  |  '
    'Left=ON  Right=OFF',
    fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../experiments/results/00_rgb_representation.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Each column = one 480-timestep window (48 minutes)')
print('Each pixel = one DWT coefficient')
print('Colors encode frequency content of the power signal')


## 12. Cross-House Comparison — House 1 vs House 2

In [ ]:
fig, axes = plt.subplots(2, len(APPLIANCE_NAMES),
                          figsize=(5*len(APPLIANCE_NAMES), 10))

for i, a in enumerate(APPLIANCE_NAMES):
    # House 1 distribution
    on1 = h1.loc[h1[f'{a}_state'] > 0, a].values
    on2 = h2.loc[h2[f'{a}_state'] > 0, a].values

    ax = axes[0, i]
    if len(on1) > 0:
        ax.hist(on1, bins=50, alpha=0.6, color='#3366CC',
                label=f'H1 mean={on1.mean():.0f}W', density=True)
    if len(on2) > 0:
        ax.hist(on2, bins=50, alpha=0.6, color='#D94040',
                label=f'H2 mean={on2.mean():.0f}W', density=True)
    ax.set_title(a.replace('_',' ').title(), fontsize=10)
    ax.legend(fontsize=8)
    if i == 0: ax.set_ylabel('Density (when ON)', fontsize=9)

    # Duty cycle comparison
    ax2 = axes[1, i]
    duty1 = h1[f'{a}_state'].mean()*100
    duty2 = h2[f'{a}_state'].mean()*100
    ax2.bar(['House 1', 'House 2'], [duty1, duty2],
            color=['#3366CC', '#D94040'], alpha=0.8)
    ax2.set_title(f'Duty cycle (%)', fontsize=9)
    ax2.set_ylabel('% time ON' if i==0 else '')
    for k, (h, d) in enumerate(zip(['H1','H2'], [duty1, duty2])):
        ax2.text(k, d+0.1, f'{d:.2f}%', ha='center', fontsize=9)

plt.suptitle('UK-DALE Cross-House Comparison: House 1 vs House 2\n'
             'Top: Power distribution when ON | Bottom: Duty cycle',
             fontsize=13)
plt.tight_layout()
plt.savefig('../experiments/results/00_cross_house_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 13. Summary Statistics Table

In [ ]:
print('='*80)
print('UK-DALE DATASET SUMMARY')
print('='*80)

for house_name, df in [('House 1', h1), ('House 2', h2)]:
    print(f'\n{house_name}:')
    print(f'  Duration: {(df.index[-1]-df.index[0]).days} days')
    print(f'  Samples:  {len(df):,} (at 6-second resolution)')
    print(f'  Aggregate: mean={df["aggregate"].mean():.1f}W '
          f'std={df["aggregate"].std():.1f}W '
          f'max={df["aggregate"].max():.0f}W')
    print()
    print(f'  {"Appliance":<22} {"Mean ON (W)":>12} {"Duty %":>8} '
          f'{"Events":>8} {"Imbalance":>10}')
    print(f'  {"-"*65}')
    for a in APPLIANCE_NAMES:
        on_vals = df.loc[df[f'{a}_state']>0, a]
        mean_on = on_vals.mean() if len(on_vals) > 0 else 0
        duty    = df[f'{a}_state'].mean()*100
        events  = int((df[f'{a}_state'].diff()==1).sum())
        ratio   = (1-duty/100)/(duty/100+1e-6)
        print(f'  {a:<22} {mean_on:>12.1f} {duty:>8.2f} '
              f'{events:>8,} {ratio:>10.0f}:1')
print()
print('Imbalance = OFF:ON ratio (higher = harder to learn)')


---
# REDD Dataset Analysis
Reference Energy Disaggregation Dataset — 6 US homes, 120V split-phase
**Appliances:** refrigerator, washer_dryer, dishwasher, microwave

In [ ]:
from preprocessing import load_redd_house, preprocess_house
from dataset import load_clean_df, save_clean_df

REDD_APPLIANCES = ['refrigerator', 'washer_dryer', 'dishwasher', 'microwave']
REDD_COLORS = {
    'refrigerator': '#2E9E5A',
    'washer_dryer':  '#E8922A',
    'dishwasher':    '#7B4FBF',
    'microwave':     '#CC3399',
}

print('Loading REDD House 1...')
try:
    cached_r = load_clean_df('REDD', 1)
    if cached_r is not None:
        redd1 = cached_r
    else:
        raw_r = load_redd_house(house=1)
        redd1 = preprocess_house(raw_r)
        save_clean_df(redd1, 'REDD', 1)
    print(f'REDD House 1: {len(redd1):,} rows | '
          f'{(redd1.index[-1]-redd1.index[0]).days} days')
    print(f'Columns: {list(redd1.columns[:8])}...')
    REDD_LOADED = True
except Exception as e:
    print(f'REDD not available: {e}')
    REDD_LOADED = False


In [ ]:
if REDD_LOADED:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    redd_apps = [c for c in redd1.columns
                 if '_state' not in c and c != 'aggregate'
                 and redd1[c].max() > 10][:4]

    for i, a in enumerate(redd_apps):
        if i >= 4: break
        ax = axes[i]
        color = list(REDD_COLORS.values())[i]

        # Power distribution when ON
        state_col = f'{a}_state'
        if state_col in redd1.columns:
            on_vals = redd1.loc[redd1[state_col]>0, a].values
        else:
            on_vals = redd1.loc[redd1[a]>10, a].values

        if len(on_vals) > 0:
            ax.hist(on_vals, bins=60, color=color, alpha=0.75,
                    edgecolor='white')
            ax.axvline(on_vals.mean(), color='black', linestyle='--',
                       label=f'Mean={on_vals.mean():.0f}W')
            ax.set_title(f'{a.replace("_"," ").title()}\n(when ON)',
                         fontsize=10)
            ax.set_xlabel('Power (W)')
            ax.legend(fontsize=9)

    plt.suptitle('REDD House 1 — Power Distributions', fontsize=13)
    plt.tight_layout()
    plt.savefig('../experiments/results/00_redd_distributions.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('REDD data not available — skipping')


---
# AMPds2 Dataset Analysis
Almanac of Minutely Power dataset — 1 Canadian home, 2 years, 1-minute resolution

In [ ]:
from preprocessing import load_ampds2, preprocess_house

print('Loading AMPds2...')
try:
    cached_a = load_clean_df('AMPds2', 1)
    if cached_a is not None:
        ampds = cached_a
    else:
        raw_a = load_ampds2()
        ampds = preprocess_house(raw_a)
        save_clean_df(ampds, 'AMPds2', 1)
    print(f'AMPds2: {len(ampds):,} rows | '
          f'{(ampds.index[-1]-ampds.index[0]).days} days')
    AMPDS_LOADED = True
except Exception as e:
    print(f'AMPds2 not available: {e}')
    AMPDS_LOADED = False


In [ ]:
if AMPDS_LOADED:
    # Aggregate overview
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))

    # Full aggregate time series
    sample = ampds['aggregate'].resample('1H').mean()
    axes[0].plot(sample.index, sample.values,
                 color='#3366CC', linewidth=0.5)
    axes[0].set_title('AMPds2 — Aggregate Power (hourly average)', fontsize=11)
    axes[0].set_ylabel('Power (W)')
    axes[0].set_xlabel('Date')

    # Power distribution
    agg_vals = ampds['aggregate'].values
    axes[1].hist(agg_vals[agg_vals>0], bins=100,
                 color='#3366CC', alpha=0.7, edgecolor='white')
    axes[1].axvline(agg_vals.mean(), color='red', linestyle='--',
                    label=f'Mean={agg_vals.mean():.0f}W')
    axes[1].set_title('AMPds2 — Aggregate Power Distribution', fontsize=11)
    axes[1].set_xlabel('Power (W)')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('../experiments/results/00_ampds2_overview.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('AMPds2 data not available — skipping')


---
# REFIT Dataset Analysis
UK Residential Energy Flexible Technologies — 20 UK homes

In [ ]:
from preprocessing import load_refit_house, preprocess_house

print('Loading REFIT House 1...')
try:
    cached_rf = load_clean_df('REFIT', 1)
    if cached_rf is not None:
        refit1 = cached_rf
    else:
        raw_rf = load_refit_house(house=1)
        refit1 = preprocess_house(raw_rf)
        save_clean_df(refit1, 'REFIT', 1)
    print(f'REFIT House 1: {len(refit1):,} rows | '
          f'{(refit1.index[-1]-refit1.index[0]).days} days')
    REFIT_LOADED = True
except Exception as e:
    print(f'REFIT not available: {e}')
    REFIT_LOADED = False


In [ ]:
if REFIT_LOADED:
    # Aggregate overview
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    sample = refit1['aggregate'].resample('1H').mean()
    axes[0].plot(sample.index, sample.values,
                 color='#3366CC', linewidth=0.5)
    axes[0].set_title('REFIT House 1 — Aggregate Power (hourly)', fontsize=11)
    axes[0].set_ylabel('Power (W)')

    agg_vals = refit1['aggregate'].values
    axes[1].hist(agg_vals[agg_vals>0], bins=100,
                 color='#3366CC', alpha=0.7, edgecolor='white')
    axes[1].axvline(agg_vals.mean(), color='red', linestyle='--',
                    label=f'Mean={agg_vals.mean():.0f}W')
    axes[1].set_title('REFIT House 1 — Power Distribution', fontsize=11)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('../experiments/results/00_refit_overview.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('REFIT data not available — skipping')


---
## Cross-Dataset Comparison
Compare aggregate power statistics across all 4 datasets.

In [ ]:
datasets = {}
datasets['UK-DALE H1'] = h1
datasets['UK-DALE H2'] = h2
if REDD_LOADED:  datasets['REDD H1']   = redd1
if AMPDS_LOADED: datasets['AMPds2']    = ampds
if REFIT_LOADED: datasets['REFIT H1']  = refit1

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

names  = list(datasets.keys())
means  = [df['aggregate'].mean()  for df in datasets.values()]
stds   = [df['aggregate'].std()   for df in datasets.values()]
maxs   = [df['aggregate'].max()   for df in datasets.values()]

bar_colors = ['#3366CC','#2196F3','#D94040','#E8922A','#2E9E5A']

# Mean power
bars = axes[0].bar(names, means,
                   color=bar_colors[:len(names)], alpha=0.85)
axes[0].set_title('Mean Aggregate Power (W)', fontsize=11)
axes[0].set_xticklabels(names, rotation=20, ha='right')
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+5,
                 f'{val:.0f}W', ha='center', fontsize=9)

# Std deviation
bars = axes[1].bar(names, stds,
                   color=bar_colors[:len(names)], alpha=0.85)
axes[1].set_title('Std Dev Aggregate Power (W)', fontsize=11)
axes[1].set_xticklabels(names, rotation=20, ha='right')
for bar, val in zip(bars, stds):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+5,
                 f'{val:.0f}W', ha='center', fontsize=9)

# Max power
bars = axes[2].bar(names, maxs,
                   color=bar_colors[:len(names)], alpha=0.85)
axes[2].set_title('Max Aggregate Power (W)', fontsize=11)
axes[2].set_xticklabels(names, rotation=20, ha='right')
for bar, val in zip(bars, maxs):
    axes[2].text(bar.get_x()+bar.get_width()/2, val+5,
                 f'{val:.0f}W', ha='center', fontsize=9)

plt.suptitle('Cross-Dataset Aggregate Power Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('../experiments/results/00_cross_dataset_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print('='*65)
print('CROSS-DATASET SUMMARY')
print('='*65)
print(f'{"Dataset":<15} {"Mean(W)":>8} {"Std(W)":>8} '
      f'{"Max(W)":>8} {"Days":>6} {"Samples":>10}')
print('-'*65)
for name, df in datasets.items():
    days = (df.index[-1]-df.index[0]).days
    print(f'{name:<15} {df["aggregate"].mean():>8.0f} '
          f'{df["aggregate"].std():>8.0f} '
          f'{df["aggregate"].max():>8.0f} '
          f'{days:>6} {len(df):>10,}')
